# Context-FID Score Presentation
## Necessary packages and functions call

- Context-FID score: A useful metric measures how well the the synthetic time series windows ”fit” into the local context of the time series

In [1]:
import os
import torch
import numpy as np
import pandas as pd
import sys
sys.path.append(os.path.join(os.path.dirname('__file__'), '../'))
from Utils.context_fid import Context_FID
from Utils.metric_utils import display_scores
from Utils.cross_correlation import CrossCorrelLoss

## Data Loading

Load original dataset and preprocess the loaded data.

In [2]:
iterations = 5

dataset_data_paths = {
    "sines": {
        "original": "../Executed/sines/samples/sine_ground_truth_24_train.npy",
        "fake": "../Executed/sines/ddpm_fake_sines.npy"
    },
    "stocks": {
        "original": "../Executed/stocks/samples/stock_norm_truth_24_train.npy",
        "fake": "../Executed/stocks/ddpm_fake_stocks.npy"
    },
    "etth": {
        "original": "../Executed/etth/samples/etth_norm_truth_24_train.npy",
        "fake": "../Executed/etth/ddpm_fake_etth.npy"
    },
    "mujoco": {
        "original": "../Executed/mujoco/samples/mujoco_norm_truth_24_train.npy",
        "fake": "../Executed/mujoco/ddpm_fake_mujoco.npy"
    },
    "energy": {
        "original": "../Executed/energy/samples/energy_norm_truth_24_train.npy",
        "fake": "../Executed/energy/ddpm_fake_energy.npy"
    },
    "fmri": {
        "original": "../Executed/fmri/samples/fMRI_norm_truth_24_train.npy",
        "fake": "../Executed/fmri/ddpm_fake_fmri.npy"
    },

}
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# # ori_data = np.load('../OUTPUT/{dataset_name}/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../toy_exp/ddpm_fake_sines.npy')

## Context-FID Score

- The Frechet Inception distance-like score is based on unsupervised time series embeddings. It is able to score the fit of the fixed length synthetic samples into their context of (often much longer) true time series.

- The lowest scoring models correspond to the best performing models in downstream tasks

In [3]:
results = []
for key in dataset_data_paths.keys():
    ori_data = np.load(dataset_data_paths[key]["original"])
    fake_data = np.load(dataset_data_paths[key]["fake"])
    context_fid_score = []
    print("Dataset", key)
    for i in range(iterations):    
        context_fid = Context_FID(ori_data[:], fake_data[:ori_data.shape[0]])
        context_fid_score.append(context_fid)
        print(f'- Iter {i}: ', 'context-fid =', context_fid)
    mean, sigma = display_scores(context_fid_score)
    results.append({"dataset": key, "mean": mean, "sigma": sigma})
results_df = pd.DataFrame(results)
results_df.to_csv("context_fid_score.csv", index=False)

Dataset sines


/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:53: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


- Iter 0:  context-fid = 0.009638519829540032
- Iter 1:  context-fid = 0.010454035381064646
- Iter 2:  context-fid = 0.007708551321400194
- Iter 3:  context-fid = 0.00762036746723499
- Iter 4:  context-fid = 0.010142865393280631
Final Score:  0.009112867878504098 ± 0.00168148053043443
Dataset stocks
- Iter 0:  context-fid = 0.14624835570422304
- Iter 1:  context-fid = 0.18897617035180778
- Iter 2:  context-fid = 0.15295510605809637
- Iter 3:  context-fid = 0.20750145237400247
- Iter 4:  context-fid = 0.2201172511112982
Final Score:  0.18315966711988557 ± 0.040554054227438616
Dataset etth
- Iter 0:  context-fid = 0.14661686148927205
- Iter 1:  context-fid = 0.13989564991980868
- Iter 2:  context-fid = 0.13241375610986722
- Iter 3:  context-fid = 0.13532341329343156
- Iter 4:  context-fid = 0.13823293413927235
Final Score:  0.13849652299033038 ± 0.006658712201078179
Dataset mujoco
- Iter 0:  context-fid = 0.012602734927731962
- Iter 1:  context-fid = 0.01112998282670535
- Iter 2:  contex

## Correlational Score

- The metric uses the absolute error of the auto-correlation estimator by real data and synthetic data as the metric to assess the temporal dependency.

- For d > 1, it uses the l1-norm of the difference between cross correlation matrices.

In [4]:
def random_choice(size, num_select=100):
    select_idx = np.random.randint(low=0, high=size, size=(num_select,))
    return select_idx

In [ ]:
results = []
for key in dataset_data_paths.keys():
    ori_data = np.load(dataset_data_paths[key]["original"])
    fake_data = np.load(dataset_data_paths[key]["fake"])

    x_real = torch.from_numpy(ori_data)
    x_fake = torch.from_numpy(fake_data)

    correlational_score = []
    size = int(x_real.shape[0] / iterations)
    print("Dataset", key)
    for i in range(iterations):
        real_idx = random_choice(x_real.shape[0], size)
        fake_idx = random_choice(x_fake.shape[0], size)
        corr = CrossCorrelLoss(x_real[real_idx, :, :], name='CrossCorrelLoss')
        loss = corr.compute(x_fake[fake_idx, :, :])
        correlational_score.append(loss.item())
        print(f'Iter {i}: ', 'cross-correlation =', loss.item())
    mean, sigma = display_scores(correlational_score)
    results.append({"dataset": key, "mean": mean, "sigma": sigma})
results_df = pd.DataFrame(results)
results_df.to_csv("correlational_score.csv", index=False)

Iter 0:  cross-correlation = 0.012645158583875299 

Iter 1:  cross-correlation = 0.0126554745801768 

Iter 2:  cross-correlation = 0.017519361878490035 

Iter 3:  cross-correlation = 0.01995443772253233 

Iter 4:  cross-correlation = 0.014149618504351619 

Final Score:  0.015384810253885217 ± 0.004019544115509883
Iter 0:  cross-correlation = 0.009543960185960732 

Iter 1:  cross-correlation = 0.0018893422236048174 

Iter 2:  cross-correlation = 0.0005323991051097487 

Iter 3:  cross-correlation = 0.014980821343328454 

Iter 4:  cross-correlation = 0.01179630721342143 

Final Score:  0.007748566014285037 ± 0.007811555181747758
Iter 0:  cross-correlation = 0.05387196007444025 

Iter 1:  cross-correlation = 0.051773543802046296 

Iter 2:  cross-correlation = 0.05508606674756794 

Iter 3:  cross-correlation = 0.036375104468495414 

Iter 4:  cross-correlation = 0.04647442567488955 

Final Score:  0.048716220153487894 ± 0.009493505988964179
Iter 0:  cross-correlation = 0.18785773286164606 

